# Dedicated Sepsis GCN

This notebook creates **new sepsis labels and a separate sepsis GCN**. It does not overwrite mortality checkpoints or metrics.

Pipeline:
1. Read `attention_embeddings_{train,val,test}.npz` and use each file's `patient_ids` as the authoritative alignment order.
2. Build a global `patientunitstayid → hospital` map from every hospital's `tabular_features.parquet`.
3. Extract sepsis positives from per-hospital `patient.csv` / `patient.parquet` and `diagnosis.csv` / `diagnosis.parquet` tables.
4. Save labels in embedding order, build hospital-restricted cosine k-NN graphs, train a dedicated GCN, select epoch and threshold using validation only, and evaluate test once.

`K_NEIGHBORS = 5` is fixed from the previous graph grid search. It was selected using mortality validation AUPRC

In [2]:
from pathlib import Path
import json
import re
import time
import copy
import warnings
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import (
    average_precision_score, roc_auc_score, accuracy_score,
    balanced_accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix
)
import torch
import torch.nn as nn
import torch.nn.functional as F

#Project paths: resilient resolution from root or subfolder
_start = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in [_start, *_start.parents] if (p / "checkpoints").exists() and (p / "processed").exists()), Path.home() / "Desktop" / "Project Hail Minor")
EMBEDDING_DIR = PROJECT_ROOT / "checkpoints" / "fusion" / "attention_embeddings"
HOSPITALS_DIR = PROJECT_ROOT / "processed" / "hospitals"
OUTPUT_DIR = PROJECT_ROOT / "checkpoints" / "fusion" / "gnn_sepsis"
LABEL_DIR = PROJECT_ROOT / "checkpoints" / "fusion"

#Training configuration
K_NEIGHBORS = 5
HIDDEN_DIM = 256
DROPOUT = 0.35
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-5
MAX_EPOCHS = 500
PATIENCE = 15
SEED = 42
THRESHOLDS = np.linspace(0.01, 0.99, 99)

for path in [EMBEDDING_DIR, HOSPITALS_DIR]:
    if not path.exists():
        raise FileNotFoundError(f"Required path does not exist: {path}")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Project root:", PROJECT_ROOT)
print("Embedding directory:", EMBEDDING_DIR)
print("Hospital directory:", HOSPITALS_DIR)
print("Output directory:", OUTPUT_DIR)
print("Device:", DEVICE)
print("k:", K_NEIGHBORS)

Project root: /Users/aryandwivedi/Desktop/Project Hail Minor
Embedding directory: /Users/aryandwivedi/Desktop/Project Hail Minor/checkpoints/fusion/attention_embeddings
Hospital directory: /Users/aryandwivedi/Desktop/Project Hail Minor/processed/hospitals
Output directory: /Users/aryandwivedi/Desktop/Project Hail Minor/checkpoints/fusion/gnn_sepsis
Device: mps
k: 5


## 1. Load embeddings and preserve patient-ID alignment

In [3]:
def normalize_patient_id(value):
    """Normalize integer-like IDs read from NPZ, CSV, or Parquet."""
    if pd.isna(value):
        return None
    try:
        return str(int(float(value)))
    except (TypeError, ValueError, OverflowError):
        return str(value).strip()


def load_embedding_split(split_name):
    path = EMBEDDING_DIR / f"attention_embeddings_{split_name}.npz"
    if not path.exists():
        raise FileNotFoundError(f"Missing embedding file: {path}")
    with np.load(path, allow_pickle=True) as data:
        required = {"embeddings", "patient_ids", "labels"}
        missing = required - set(data.files)
        if missing:
            raise KeyError(f"{path.name} missing NPZ keys: {sorted(missing)}")
        X = np.asarray(data["embeddings"], dtype=np.float32)
        ids = np.asarray([normalize_patient_id(v) for v in data["patient_ids"]], dtype=object)
        # The existing labels are mortality labels. They are intentionally not used as sepsis targets.
    if X.ndim != 2 or len(X) != len(ids):
        raise ValueError(f"Invalid embedding/ID shapes in {path.name}: X={X.shape}, ids={ids.shape}")
    if any(v is None for v in ids):
        raise ValueError(f"Null patient IDs found in {path.name}")
    if len(set(ids.tolist())) != len(ids):
        raise ValueError(f"Duplicate patient IDs found in {path.name}")
    if not np.isfinite(X).all():
        raise ValueError(f"Non-finite embedding values found in {path.name}")
    return X, ids


DATA = {}
for split_name in ("train", "val", "test"):
    X, ids = load_embedding_split(split_name)
    DATA[split_name] = {"X": X, "ids": ids}
    print(f"{split_name:>5}: X={X.shape}, unique IDs={len(set(ids.tolist())):,}")

# Ensure split IDs are disjoint. If this fails, stop rather than silently evaluate on overlap.
for a, b in [("train", "val"), ("train", "test"), ("val", "test")]:
    overlap = set(DATA[a]["ids"].tolist()) & set(DATA[b]["ids"].tolist())
    if overlap:
        raise ValueError(f"Patient-level split overlap between {a} and {b}: {len(overlap)} IDs; examples={list(overlap)[:5]}")
print("Embedding splits are patient-disjoint.")

train: X=(153831, 64), unique IDs=153,831
  val: X=(24186, 64), unique IDs=24,186
 test: X=(22809, 64), unique IDs=22,809
Embedding splits are patient-disjoint.


## 2. Build a global patient-to-hospital map

In [4]:
patient_to_hospital = {}
hospital_dirs = sorted(HOSPITALS_DIR.glob("hospital_*"))
if not hospital_dirs:
    raise FileNotFoundError(f"No hospital_* folders found under {HOSPITALS_DIR}")

for hospital_dir in hospital_dirs:
    parquet_path = hospital_dir / "tabular" / "tabular_features.parquet"
    if not parquet_path.exists():
        warnings.warn(f"Missing {parquet_path}; skipped for hospital mapping")
        continue
    ids = pd.read_parquet(parquet_path, columns=["patientunitstayid"])["patientunitstayid"].dropna().unique()
    for raw_id in ids:
        pid = normalize_patient_id(raw_id)
        old_hospital = patient_to_hospital.get(pid)
        if old_hospital is not None and old_hospital != hospital_dir.name:
            raise ValueError(f"Patient {pid} appears in both {old_hospital} and {hospital_dir.name}")
        patient_to_hospital[pid] = hospital_dir.name

all_embedding_ids = set().union(*(set(d["ids"].tolist()) for d in DATA.values()))
unmapped = all_embedding_ids - set(patient_to_hospital)
print("Hospital folders:", len(hospital_dirs))
print("Unique patient IDs mapped:", len(patient_to_hospital))
print("Unique embedding patient IDs:", len(all_embedding_ids))
print("Unmapped embedding IDs:", len(unmapped))
if unmapped:
    raise ValueError(f"Could not map {len(unmapped)} embedding IDs to hospitals; examples={list(unmapped)[:10]}")

Hospital folders: 208
Unique patient IDs mapped: 200859
Unique embedding patient IDs: 200826
Unmapped embedding IDs: 0


## 3. Extract sepsis labels from per-hospital diagnosis sources

In [5]:
# Criteria from the implementation plan. A patient is positive if ANY source matches.
ICD9_EXACT_CODES = {"99591", "99592", "78552"}
ICD9_PREFIXES = ("038",)
SEPSIS_KEYWORDS = ("seps", "septic")
SUPPORTED_SUFFIXES = {".csv", ".parquet"}


def read_source_table(path):
    if path.suffix.lower() == ".parquet":
        return pd.read_parquet(path)
    if path.suffix.lower() == ".csv":
        return pd.read_csv(path, low_memory=False)
    raise ValueError(f"Unsupported table format: {path}")


def find_source_tables(hospital_dir, stem):
    # Scan only named source tables, never feature/time-series parquet files.
    matches = [p for p in hospital_dir.rglob("*")
               if p.is_file() and p.stem.lower() == stem.lower()
               and p.suffix.lower() in SUPPORTED_SUFFIXES]
    # Prefer files under tabular/, then deterministic ordering.
    return sorted(matches, key=lambda p: ("tabular" not in [part.lower() for part in p.parts], str(p)))


def get_col(df, name):
    lookup = {str(c).lower(): c for c in df.columns}
    return lookup.get(name.lower())


def text_matches_sepsis(value):
    if pd.isna(value):
        return False
    text = str(value).strip().lower()
    return any(keyword in text for keyword in SEPSIS_KEYWORDS)


def icd9_matches_sepsis(value):
    if pd.isna(value):
        return False
    # Ignore punctuation and whitespace: e.g. 038.9 -> 0389; 995.92 -> 99592.
    code = re.sub(r"[^a-zA-Z0-9]", "", str(value)).upper()
    return code in ICD9_EXACT_CODES or any(code.startswith(prefix) for prefix in ICD9_PREFIXES)

positive_ids = set()
source_counts = Counter()
read_files = []
problems = []

for hospital_dir in hospital_dirs:
    # Patient table: use apacheadmissiondx.
    patient_tables = find_source_tables(hospital_dir, "patient")
    for path in patient_tables:
        try:
            df = read_source_table(path)
            id_col = get_col(df, "patientunitstayid")
            dx_col = get_col(df, "apacheadmissiondx")
            if id_col is None or dx_col is None:
                problems.append((str(path), f"required columns not found; ID={id_col}, apacheadmissiondx={dx_col}"))
                continue
            matched = df[dx_col].map(text_matches_sepsis)
            ids = df.loc[matched, id_col].dropna().map(normalize_patient_id)
            new_ids = set(ids) - positive_ids
            positive_ids.update(ids)
            source_counts["apacheadmissiondx_positive_rows"] += int(matched.sum())
            source_counts["apacheadmissiondx_positive_unique_patients"] += len(new_ids)
            read_files.append(str(path))
        except Exception as exc:
            problems.append((str(path), repr(exc)))

    # Diagnosis table: diagnosisstring and ICD-9 code.
    diagnosis_tables = find_source_tables(hospital_dir, "diagnosis")
    for path in diagnosis_tables:
        try:
            df = read_source_table(path)
            id_col = get_col(df, "patientunitstayid")
            if id_col is None:
                problems.append((str(path), "patientunitstayid column not found"))
                continue
            string_col = get_col(df, "diagnosisstring")
            code_col = get_col(df, "icd9code")
            if string_col is None and code_col is None:
                problems.append((str(path), "neither diagnosisstring nor icd9code column found"))
                continue
            matched = pd.Series(False, index=df.index)
            if string_col is not None:
                matched |= df[string_col].map(text_matches_sepsis)
            if code_col is not None:
                matched |= df[code_col].map(icd9_matches_sepsis)
            ids = df.loc[matched, id_col].dropna().map(normalize_patient_id)
            new_ids = set(ids) - positive_ids
            positive_ids.update(ids)
            source_counts["diagnosis_positive_rows"] += int(matched.sum())
            source_counts["diagnosis_positive_unique_incremental_patients"] += len(new_ids)
            read_files.append(str(path))
        except Exception as exc:
            problems.append((str(path), repr(exc)))

print("Source files successfully used:", len(set(read_files)))
print("Positive patient IDs found across sources:", len(positive_ids))
print("Matching rows/counts:", dict(source_counts))
print("Source-table issues:", len(problems))
for item in problems[:15]:
    print("  ", item)

if not read_files:
    raise RuntimeError(
        "No usable patient/diagnosis source tables were found. Check that patient.csv and diagnosis.csv "
        "or their Parquet equivalents exist under processed/hospitals/hospital_*/. "
        "The notebook will not silently generate all-negative labels."
    )
if not positive_ids:
    raise RuntimeError(
        "No sepsis-positive patients matched the supplied criteria. Inspect the source-table issues and actual column values before training."
    )

# Count how many embedding patients can be labelled from the found positive IDs.
for split_name, split_data in DATA.items():
    split_ids = split_data["ids"]
    y = np.fromiter((int(pid in positive_ids) for pid in split_ids), dtype=np.int64, count=len(split_ids))
    DATA[split_name]["y"] = y
    label_path = LABEL_DIR / f"sepsis_labels_{split_name}.npy"
    np.save(label_path, y)
    print(f"{split_name:>5}: patients={len(y):,}, positives={int(y.sum()):,}, prevalence={y.mean():.4%}, saved={label_path}")

# Save a mapping audit so label-generation provenance is preserved.
label_summary = {
    "label_definition": {
        "diagnosis_keywords": list(SEPSIS_KEYWORDS),
        "icd9_prefixes": list(ICD9_PREFIXES),
        "icd9_exact_normalized_codes": sorted(ICD9_EXACT_CODES),
        "apache_column": "apacheadmissiondx",
        "diagnosis_columns": ["diagnosisstring", "icd9code"],
    },
    "positive_patient_ids_found": len(positive_ids),
    "source_files_read": sorted(set(read_files)),
    "source_counts": dict(source_counts),
    "source_table_issues": problems,
    "splits": {
        split: {"n": int(len(DATA[split]["y"])), "positives": int(DATA[split]["y"].sum()), "prevalence": float(DATA[split]["y"].mean())}
        for split in DATA
    },
    "note": "Diagnosis-based proxy phenotype; this does not establish sepsis onset timing.",
}
with open(OUTPUT_DIR / "sepsis_label_extraction_summary.json", "w") as f:
    json.dump(label_summary, f, indent=2)
print("Label audit saved to:", OUTPUT_DIR / "sepsis_label_extraction_summary.json")

Source files successfully used: 416
Positive patient IDs found across sources: 32142
Matching rows/counts: {'apacheadmissiondx_positive_rows': 23149, 'apacheadmissiondx_positive_unique_patients': 23149, 'diagnosis_positive_rows': 160347, 'diagnosis_positive_unique_incremental_patients': 8993}
Source-table issues: 0
train: patients=153,831, positives=24,477, prevalence=15.9116%, saved=/Users/aryandwivedi/Desktop/Project Hail Minor/checkpoints/fusion/sepsis_labels_train.npy
  val: patients=24,186, positives=3,919, prevalence=16.2036%, saved=/Users/aryandwivedi/Desktop/Project Hail Minor/checkpoints/fusion/sepsis_labels_val.npy
 test: patients=22,809, positives=3,746, prevalence=16.4233%, saved=/Users/aryandwivedi/Desktop/Project Hail Minor/checkpoints/fusion/sepsis_labels_test.npy
Label audit saved to: /Users/aryandwivedi/Desktop/Project Hail Minor/checkpoints/fusion/gnn_sepsis/sepsis_label_extraction_summary.json


## 4. Build hospital-restricted k-NN adjacency matrices

In [6]:
def build_hospital_knn_adjacency(X, patient_ids, patient_hospital, k):
    """Build symmetric, cosine-weighted k-NN graph without cross-hospital edges."""
    n = len(patient_ids)
    if X.shape[0] != n:
        raise ValueError(f"X/ID length mismatch: {X.shape[0]} vs {n}")
    if k < 1:
        raise ValueError("k must be >= 1")

    groups = defaultdict(list)
    for i, pid in enumerate(patient_ids):
        hospital = patient_hospital.get(pid)
        if hospital is None:
            raise ValueError(f"Patient {pid} has no hospital mapping")
        groups[hospital].append(i)

    row_idx, col_idx, weights = [], [], []
    graph_search_X = X.astype(np.float32, copy=False)

    for hospital, global_indices in groups.items():
        if len(global_indices) < 2:
            continue
        indices = np.asarray(global_indices, dtype=np.int64)
        X_h = graph_search_X[indices]
        k_eff = min(k, len(indices) - 1)
        nn = NearestNeighbors(n_neighbors=k_eff + 1, metric="cosine", algorithm="brute", n_jobs=-1)
        nn.fit(X_h)
        distances, neighbors = nn.kneighbors(X_h, return_distance=True)

        for local_i in range(len(indices)):
            added = 0
            for dist, local_j in zip(distances[local_i], neighbors[local_i]):
                if local_j == local_i:
                    continue
                # Cosine similarity as edge weight. Clip tiny/negative values to keep weights positive.
                sim = max(1.0 - float(dist), 1e-6)
                row_idx.append(int(indices[local_i]))
                col_idx.append(int(indices[local_j]))
                weights.append(sim)
                added += 1
                if added == k_eff:
                    break

    A = sparse.coo_matrix((np.asarray(weights, dtype=np.float32),
                           (np.asarray(row_idx, dtype=np.int64), np.asarray(col_idx, dtype=np.int64))),
                          shape=(n, n), dtype=np.float32).tocsr()
    # Symmetrize: if either node selects the other, retain the stronger edge.
    A = A.maximum(A.T).tocsr()
    A.setdiag(1.0)
    A.eliminate_zeros()

    degree = np.asarray(A.sum(axis=1)).ravel()
    d_inv_sqrt = np.zeros_like(degree, dtype=np.float32)
    nonzero = degree > 0
    d_inv_sqrt[nonzero] = np.power(degree[nonzero], -0.5)
    D = sparse.diags(d_inv_sqrt)
    A_norm = (D @ A @ D).tocoo()

    indices_t = torch.from_numpy(np.vstack([A_norm.row, A_norm.col]).astype(np.int64))
    values_t = torch.from_numpy(A_norm.data.astype(np.float32))
    A_torch = torch.sparse_coo_tensor(indices_t, values_t, size=A_norm.shape).coalesce()
    return A_torch, int(A.nnz), {h: len(ix) for h, ix in groups.items()}


for split_name in ("train", "val", "test"):
    start = time.time()
    A, nnz, group_sizes = build_hospital_knn_adjacency(
        DATA[split_name]["X"], DATA[split_name]["ids"], patient_to_hospital, K_NEIGHBORS
    )
    DATA[split_name]["adj"] = A.to(DEVICE)
    print(f"{split_name:>5}: graph={tuple(A.shape)}, entries incl. self-loops={nnz:,}, "
          f"hospitals represented={len(group_sizes)}, seconds={time.time()-start:.1f}")

# Graph size and hospital separation checks.
for split_name in ("train", "val", "test"):
    A = DATA[split_name]["adj"]
    assert A.shape[0] == len(DATA[split_name]["y"])
    assert A.is_coalesced()
print("Graph shape checks passed.")

/var/folders/km/bh6401hj3bl7j89v4k3ql4r00000gn/T/ipykernel_10099/1880681657.py:60: UserWarning: Sparse invariant checks are implicitly disabled. Memory errors (e.g. SEGFAULT) will occur when operating on a sparse tensor which violates the invariants, but checks incur performance overhead. To silence this warning, explicitly opt in or out. See `torch.sparse.check_sparse_tensor_invariants.__doc__` for guidance.  (Triggered internally at /Users/runner/work/pytorch/pytorch/aten/src/ATen/Context.cpp:855.)
  A_torch = torch.sparse_coo_tensor(indices_t, values_t, size=A_norm.shape).coalesce()


train: graph=(153831, 153831), entries incl. self-loops=1,301,989, hospitals represented=144, seconds=5.3
  val: graph=(24186, 24186), entries incl. self-loops=202,504, hospitals represented=31, seconds=0.8
 test: graph=(22809, 22809), entries incl. self-loops=192,001, hospitals represented=32, seconds=0.8
Graph shape checks passed.


## 5. Train the dedicated Sepsis GCN

In [7]:
class SepsisGCN(nn.Module):
    def __init__(self, input_dim=64, hidden_dim=256, dropout=0.35):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.sepsis_head = nn.Linear(hidden_dim, 1)
        self.dropout = float(dropout)

    def forward(self, x, adj):
        h = torch.sparse.mm(adj, x)
        h = F.relu(self.fc1(h))
        h = F.dropout(h, p=self.dropout, training=self.training)
        h = torch.sparse.mm(adj, h)
        h = F.relu(self.fc2(h))
        h = F.dropout(h, p=self.dropout, training=self.training)
        return self.sepsis_head(h).squeeze(-1)


X_tensors = {}
y_tensors = {}
for split_name in ("train", "val", "test"):
    X_tensors[split_name] = torch.as_tensor(DATA[split_name]["X"], dtype=torch.float32, device=DEVICE)
    y_tensors[split_name] = torch.as_tensor(DATA[split_name]["y"], dtype=torch.float32, device=DEVICE)

n_pos = int(DATA["train"]["y"].sum())
n_neg = int(len(DATA["train"]["y"]) - n_pos)
if n_pos == 0:
    raise ValueError("Training split has no positive sepsis labels")
pos_weight = torch.tensor([n_neg / n_pos], dtype=torch.float32, device=DEVICE)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

model = SepsisGCN(
    input_dim=DATA["train"]["X"].shape[1],
    hidden_dim=HIDDEN_DIM,
    dropout=DROPOUT,
).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

best_val_auprc = -np.inf
best_epoch = None
best_state = None
patience_counter = 0
history = []
start_training = time.time()

for epoch in range(1, MAX_EPOCHS + 1):
    model.train()
    optimizer.zero_grad(set_to_none=True)
    train_logits = model(X_tensors["train"], DATA["train"]["adj"])
    train_loss = criterion(train_logits, y_tensors["train"])
    train_loss.backward()
    optimizer.step()

    model.eval()
    with torch.no_grad():
        val_logits = model(X_tensors["val"], DATA["val"]["adj"])
        val_probs = torch.sigmoid(val_logits).detach().cpu().numpy()
        val_auprc = float(average_precision_score(DATA["val"]["y"], val_probs))
        val_loss = float(criterion(val_logits, y_tensors["val"]).item())

    row = {
        "epoch": epoch,
        "train_loss": float(train_loss.item()),
        "val_loss": val_loss,
        "val_auprc": val_auprc,
    }
    history.append(row)
    print(f"Epoch {epoch:03d} | train loss={row['train_loss']:.5f} | "
          f"val loss={val_loss:.5f} | val AUPRC={val_auprc:.5f}")

    if val_auprc > best_val_auprc:
        best_val_auprc = val_auprc
        best_epoch = epoch
        best_state = copy.deepcopy(model.state_dict())
        patience_counter = 0
        torch.save({
            "model_state_dict": best_state,
            "input_dim": DATA["train"]["X"].shape[1],
            "hidden_dim": HIDDEN_DIM,
            "dropout": DROPOUT,
            "k_neighbors": K_NEIGHBORS,
            "best_epoch": best_epoch,
            "best_val_auprc": best_val_auprc,
        }, OUTPUT_DIR / "gcn_sepsis.pt")
    else:
        patience_counter += 1

    if patience_counter >= PATIENCE:
        print(f"Early stopping at epoch {epoch}; best epoch was {best_epoch}.")
        break

if best_state is None:
    raise RuntimeError("Training did not produce a best model state")
model.load_state_dict(best_state)
print(f"\nBest epoch: {best_epoch}; best validation AUPRC: {best_val_auprc:.5f}")
print(f"Training time: {time.time()-start_training:.1f} seconds")
print("Sepsis checkpoint:", OUTPUT_DIR / "gcn_sepsis.pt")

Epoch 001 | train loss=1.16043 | val loss=1.14610 | val AUPRC=0.34591
Epoch 002 | train loss=1.11801 | val loss=1.12352 | val AUPRC=0.34999
Epoch 003 | train loss=1.07983 | val loss=1.10138 | val AUPRC=0.35110
Epoch 004 | train loss=1.04323 | val loss=1.07960 | val AUPRC=0.35169
Epoch 005 | train loss=1.00715 | val loss=1.05892 | val AUPRC=0.35239
Epoch 006 | train loss=0.97264 | val loss=1.04072 | val AUPRC=0.35317
Epoch 007 | train loss=0.93964 | val loss=1.02652 | val AUPRC=0.35412
Epoch 008 | train loss=0.91028 | val loss=1.01761 | val AUPRC=0.35524
Epoch 009 | train loss=0.88515 | val loss=1.01460 | val AUPRC=0.35652
Epoch 010 | train loss=0.86572 | val loss=1.01700 | val AUPRC=0.35776
Epoch 011 | train loss=0.85072 | val loss=1.02307 | val AUPRC=0.35913
Epoch 012 | train loss=0.84030 | val loss=1.03023 | val AUPRC=0.36048
Epoch 013 | train loss=0.83240 | val loss=1.03586 | val AUPRC=0.36166
Epoch 014 | train loss=0.82711 | val loss=1.03791 | val AUPRC=0.36284
Epoch 015 | train lo

## 6. Select threshold using validation only, then evaluate test once

In [8]:
model.eval()
with torch.no_grad():
    val_probs = torch.sigmoid(model(X_tensors["val"], DATA["val"]["adj"])).cpu().numpy()

val_y = DATA["val"]["y"].astype(int)
threshold_rows = []
for threshold in THRESHOLDS:
    pred = (val_probs >= threshold).astype(int)
    threshold_rows.append((
        float(threshold),
        float(f1_score(val_y, pred, zero_division=0)),
        float(precision_score(val_y, pred, zero_division=0)),
        float(recall_score(val_y, pred, zero_division=0)),
    ))

# Maximize validation F1; tie-break by higher precision, then higher threshold.
selected_threshold, val_best_f1, val_precision, val_recall = max(
    threshold_rows, key=lambda x: (x[1], x[2], x[0])
)
print(f"Selected validation threshold: {selected_threshold:.4f}")
print(f"Validation F1 at threshold: {val_best_f1:.4f}")
print(f"Validation precision / recall: {val_precision:.4f} / {val_recall:.4f}")

# Test evaluation happens only after model/threshold selection is complete.
with torch.no_grad():
    test_probs = torch.sigmoid(model(X_tensors["test"], DATA["test"]["adj"])).cpu().numpy()

test_y = DATA["test"]["y"].astype(int)
test_pred = (test_probs >= selected_threshold).astype(int)
cm = confusion_matrix(test_y, test_pred, labels=[0, 1])
tn, fp, fn, tp = cm.ravel()
specificity = float(tn / (tn + fp)) if (tn + fp) else float("nan")
metrics = {
    "device": str(DEVICE),
    "k_neighbors": int(K_NEIGHBORS),
    "hidden_dim": int(HIDDEN_DIM),
    "dropout": float(DROPOUT),
    "learning_rate": float(LEARNING_RATE),
    "weight_decay": float(WEIGHT_DECAY),
    "pos_weight": float(n_neg / n_pos),
    "best_epoch": int(best_epoch),
    "validation_auprc": float(best_val_auprc),
    "validation_threshold": float(selected_threshold),
    "validation_f1_at_threshold": float(val_best_f1),
    "test_auroc": float(roc_auc_score(test_y, test_probs)),
    "test_auprc": float(average_precision_score(test_y, test_probs)),
    "test_accuracy": float(accuracy_score(test_y, test_pred)),
    "test_balanced_accuracy": float(balanced_accuracy_score(test_y, test_pred)),
    "test_precision": float(precision_score(test_y, test_pred, zero_division=0)),
    "test_recall_sensitivity": float(recall_score(test_y, test_pred, zero_division=0)),
    "test_f1": float(f1_score(test_y, test_pred, zero_division=0)),
    "test_specificity": specificity,
    "test_confusion_matrix": cm.tolist(),
    "test_n": int(len(test_y)),
    "test_positives": int(test_y.sum()),
    "test_prevalence": float(test_y.mean()),
    "graph_entries_including_self_loops": {
        split: int(DATA[split]["adj"]._nnz()) for split in ("train", "val", "test")
    },
}

with open(OUTPUT_DIR / "metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)
with open(OUTPUT_DIR / "training_history.json", "w") as f:
    json.dump(history, f, indent=2)
np.save(OUTPUT_DIR / "test_probabilities.npy", test_probs)
np.save(OUTPUT_DIR / "test_predictions.npy", test_pred)

print("\nSEPSIS GCN TEST RESULTS")
for key, value in metrics.items():
    print(f"{key}: {value}")
print("\nSaved outputs to:", OUTPUT_DIR)
print("Mortality checkpoints are not written by this notebook.")

Selected validation threshold: 0.6300
Validation F1 at threshold: 0.4204
Validation precision / recall: 0.3297 / 0.5800

SEPSIS GCN TEST RESULTS
device: mps
k_neighbors: 5
hidden_dim: 256
dropout: 0.35
learning_rate: 0.001
weight_decay: 1e-05
pos_weight: 5.284716264248069
best_epoch: 17
validation_auprc: 0.3647827269299706
validation_threshold: 0.63
validation_f1_at_threshold: 0.42038098760865544
test_auroc: 0.834306675096112
test_auprc: 0.48800487228709394
test_accuracy: 0.7593055372879126
test_balanced_accuracy: 0.7558351843113061
test_precision: 0.38165038002171553
test_recall_sensitivity: 0.7506673785371063
test_f1: 0.5060284326075221
test_specificity: 0.761002990085506
test_confusion_matrix: [[14507, 4556], [934, 2812]]
test_n: 22809
test_positives: 3746
test_prevalence: 0.1642334166337849
graph_entries_including_self_loops: {'train': 1301989, 'val': 202504, 'test': 192001}

Saved outputs to: /Users/aryandwivedi/Desktop/Project Hail Minor/checkpoints/fusion/gnn_sepsis
Mortality ch

## Interpretation and safeguards

- `test_auprc` is the main ranking metric for this imbalanced outcome; compare it with test prevalence as a no-skill baseline.
- `test_recall_sensitivity` and `test_specificity` describe the operating point determined from validation F1.
- The criteria use diagnosis strings and ICD-9/APACHE admission diagnoses. They may label sepsis at any point during a stay and do not prove onset before the model's prediction window.
- The notebook uses the existing patient-level embedding splits. Hospitals can therefore appear in more than one split. This is not a held-out-hospital evaluation.
- The notebook writes only to `checkpoints/fusion/gnn_sepsis/` and `checkpoints/fusion/sepsis_labels_*.npy`; it does not write to mortality checkpoint files.
- The claimed prevalence or a target AUPRC threshold should be measured from the resulting labels and metrics, not assumed in advance.